## BM25

Mentre con VSM abbiamo usato uno score del tipo
$$\text{score}(d, q) = \text{cos}(d, q)$$
cioè una misura di somiglianza **geometrica** tra vettore query e vettore documento, con BM25 il punto di vista cambia.

Non si normalizzano più esplicitamente i vettori con la norma 2, ma si assegna ad ogni documento uno score come somma dei contributi dei termini presenti nel documento che appartengono anche alla query:
$$\text{score}(d, q) = \sum_{t \in q} \text{contributo}(t, d)$$
In particolare ogni contributo dipende da:
- quanto il termine è raro nella collezione (IDF)
- quante volte compare nel documento (TF), ma con saturazione
- quanto è lungo il documento 

L'idea intuitiva di BM25 è conservare alcune delle intuizioni di TF-IDF, ma modificarle con una motivazione probabilistica e con una gestione più controllata e esplicita della lunghezza del documento e della frequenza dei termini.

Infatti come TF-IDF, anche BM25 assegna più importanza ai termini rari, però a differenza sua:
- non usa una crescita logaritmica per la TF e introduce una saturazione vera e propria della TF, introducendo un parametro apposito
- normalizza rispetto alla lunghezza del documento in modo più controllato sfruttando un paramentro apposito

Nel laboratorio stesso preprocessing visto in precedenza, per poter confrontare facilmente i modelli (quindi rimozione header, lowercase, normalizzazione numeri, rimozione punteggiatura stopword e token troppo corti, stemming). Importante sottolineare che l'obiettivo non è costruire il mglior preprocessing del mondo, ma avere una rappresentazione facile e coerente per confrontare i modelli. 

Sempre come nel laboratorio precedente, useremo un dizionario docID --> lista di token nel documento. Questa rappresentazione sarà sufficiente per costruire le statistiche richieste per calcolare gli score BM25 e LM.

```python
def build_tokenized_documents(documents):

    tokenized_documents = {}

    for doc_id, text in enumerate(documents):
        tokenized_documents[doc_id] = preprocess(text, is_query=False)

    return tokenized_documents
```

Tra le statistiche fondamentali che ci servono per BM25 e LM abbiamo:
- $L_d$: lunghezza del documento $d$ (in termini di numero di token)
- $L_{avg}$: lunghezza media dei documenti nella collezione
- $df_t$: document frequency del termine $t$, cioè in quanti documenti compare il termine $t$
- $cf_t$: collection frequency del termine $t$, cioè in quanti documenti compare il termine $t$

Di seguito vediamo un codice che calcola queste statistiche e le inserisce in un dizionario. 

Per quanto riguarda il calcolo di document_frequency e collection_frequency: Counter[tokens] restituisce un dizionario in cui le chiavi sono i token e i valori le frequenze di quei token nel documento. Quindi scorrendo i token del dizionario possiamo aggiornare sia la document frequency (incrementando di 1 per ogni termine che compare nel documento) sia la collection frequency (incrementando del numero di occorrenze del termine nel documento).

```python
def compute_collection_statistics(tokenized_documents):
    N = len(tokenized_documents)

    doc_lengths = {}
    term_frequencies = {}
    document_frequency = defaultdict(int)
    collection_frequency = defaultdict(int)

    for doc_id, tokens in tokenized_documents.items():
        doc_lengths[doc_id] = len(tokens)
        tf = Counter(tokens)
        term_frequencies[doc_id] = tf

        for term, count in tf.items():
            document_frequency[term] += 1
            collection_frequency[term] += count

    avg_doc_len = sum(doc_lengths.values()) / N
    total_tokens = sum(collection_frequency.values())
    vocabulary = sorted(collection_frequency.keys())

    return {
        "N": N,
        "doc_lengths": doc_lengths,
        "avg_doc_len": avg_doc_len,
        "term_frequencies": term_frequencies,
        "document_frequency": dict(document_frequency),
        "collection_frequency": dict(collection_frequency),
        "vocabulary": vocabulary,
        "total_tokens": total_tokens,
    }
```

In generale un termine può avere:
- alto cf e alto df --> termine molto diffuso
- alto cf e basso df --> termine molto concentrato in pochi documenti (appare tenzenzialmente poco in molti documenti ma molto in pochi documenti)
- basso cf e basso df --> termine raro

Implementazione di BM25: si utilizzerà la seguente formula:
$$BM25(d, q) = \sum_{t \in q^*} IDF(t) \cdot \frac{tf_{t,d} \cdot (k_1 + 1)}{tf_{t,d} + k_1 \cdot ((1 - b) + b \frac{L_d}{L_{avg}})}$$
dove q\* indica l'insieme di termini distinti della query.

Riguardo l'IDF, invece di usare la formula che abbiamo visto $IDF(t) = \log (1 + \frac{N}{df_t})$, usiamo una variante che comunque evita valori negativi ma abbassa ancora di più i termini presenti in molti documenti:
$$IDF(t) = \log \left(1 + \frac{N - df_t + 0.5}{df_t + 0.5}\right)$$

```python
def bm25_idf(term, document_frequency, N):
    df = document_frequency.get(term, 0)
    if df == 0:
        return 0
    return math.log(1 + (N - df + 0.5) / (df + 0.5))
```

### **Analisi del parametro $k_1$**
Il parametro $k_1$ controlla la saturazione della TF. La parte della formula BM25 che riguarda la TF è:
$$\frac{tf_{t,d} \cdot (k_1 + 1)}{tf_{t,d} + k_1 B}$$
dove 
$$B = (1 - b) + b \frac{L_d}{L_{avg}}$$
Consideriamo per semplicità un documento con lunghezza pari alla lunghezza media, quindi $B = 1$. In questo caso la formula si semplifica a:
$$\frac{tf_{t,d} \cdot (k_1 + 1)}{tf_{t,d} + k_1}$$
Osserviamo in questo caso cosa succede al variare di $k_1$:
- se $k_1 = 0$ --> la formula si riduce a $\frac{tf_{t,d} \cdot 1}{tf_{t,d} + 0} = 1$ se $tf_{t,d} > 0$, altrimenti 0. Quindi in questo caso conta solo se il termine è presente o meno per incrementare lo score
- se $k_1$ è piccolo --> la saturazione al valore $k_1 + 1$ avviene molto rapidamente
- se $k_1$ è grande --> la saturazione avviene più lentamente, quindi la TF cresce più linearmente e le occorrenze aggiuntive continuano ad avere effetto più a lungo
- per $k_1 \to \infty$ --> la formula si semplifica a $tf_{t,d}$, quindi non c'è saturazione e la TF cresce linearmente senza limiti

La saturazione converge al valore $k_1 + 1$ all'aumentare di $tf_{t,d}$ (per vederlo basta mettere nella formula $tf_{t,d} \to \infty$).

<img src="img/k_1.png" alt="BM25 k1" width="400">

Questa saturazione serve a far sì che la prima occorrenza di un termine è molto importante, poi le occorrenze successive continuando ad aumentare lo score, ma con un impatto decrescente fino a saturazione.

### Analisi del parametro $b$
BM25 come detto non usa solo la frequenza del termine nel documento, ma tiene conto anche della sua lunghezza. La componente che gestisce la lunghezza del documento è:
$$B_d = (1 - b) + b \frac{L_d}{L_{avg}}$$
dove $L_d$ è la lunghezza del documento $d$ e $L_{avg}$ è la lunghezza media dei documenti nella collezione.

In particolare la formula BM25 contiene $B_d$ al denominatore, per cui se un documento è più lungo della media ($L_d > L_{avg}$) allora $B_d > 1$ e quindi lo score del documento viene penalizzato, mentre se un documento è più corto della media ($L_d < L_{avg}$) allora $B_d < 1$ e quindi lo score del documento viene aumentato.

In questo modo BM25 cerca di evitare che i documenti più lunghi siano favoriti solo perché contengono più termini.

Nel seguente grafico confrontiamo tre documenti con lunghezze diverse (uno corto con $|d| = 50$, uno medio con $|d| = 100$, uno lungo con $|d| = 200$) e un valore di $b$ pari a 0.75. A parità di TF, si osserva come il documento più corto riceva contributo maggiore, quello medio in mezzo e quello lungo contributo minore.

<img src="img/b.png" alt="BM25 b" width="400">

```python
class BM25:
    """
    Implementazione didattica di BM25.

    Parameters
    ----------
    k1 : float
        Controlla la saturazione della term frequency.
    b : float
        Controlla la normalizzazione della lunghezza del documento.
    """

    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b

    def fit(self, tokenized_documents):
        """
        Calcola le statistiche della collezione.

        Parameters
        ----------
        tokenized_documents : dict[int, list[str]]
            Dizionario doc_id -> lista di token.

        Returns
        -------
        self
        """
        self.tokenized_documents = tokenized_documents
        self.N = len(tokenized_documents)

        self.doc_len = {}
        self.term_frequencies = {}
        self.document_frequency = defaultdict(int)

        for doc_id, tokens in tokenized_documents.items():
            self.doc_len[doc_id] = len(tokens)

            tf = Counter(tokens)
            self.term_frequencies[doc_id] = tf

            for term in tf:
                self.document_frequency[term] += 1

        self.avg_doc_len = sum(self.doc_len.values()) / self.N

        return self

    def idf(self, term):
        """
        Calcola una versione non negativa dell'idf probabilistico di BM25.

        Usiamo:

        $$
        idf(t) = log(1 + (N - df_t + 0.5)/(df_t + 0.5))
        $$

        Questa variante evita valori negativi per termini molto frequenti.
        """
        df = self.document_frequency.get(term, 0)

        if df == 0:
            return 0.0

        return math.log(1 + (self.N - df + 0.5) / (df + 0.5))

    def score_term(self, term, doc_id):
        """
        Calcola il contributo di un singolo termine allo score BM25 di un documento.
        """
        tf = self.term_frequencies[doc_id].get(term, 0)

        if tf == 0:
            return 0.0

        idf = self.idf(term)
        doc_len = self.doc_len[doc_id]

        B = (1 - self.b) + self.b * (doc_len / self.avg_doc_len)

        tf_component = ((self.k1 + 1) * tf) / (tf + self.k1 * B)

        return idf * tf_component

    def score(self, query_tokens, doc_id):
        """
        Calcola lo score BM25 di un documento rispetto a una query tokenizzata.

        In questa versione semplice consideriamo ogni termine distinto della query una sola volta.
        """
        score = 0.0

        for term in sorted(set(query_tokens)):
            score += self.score_term(term, doc_id)

        return score

    def rank(self, query, top_k=10):
        """
        Restituisce i top-k documenti per una query testuale.

        Parameters
        ----------
        query : str
            Query in linguaggio naturale.
        top_k : int
            Numero di risultati da restituire.

        Returns
        -------
        tuple
            (query_tokens, ranked_results)

        dove ranked_results è una lista di tuple:
        (doc_id, score)
        """
        query_tokens = sorted(set(preprocess(query, is_query=True)))

        results = []

        for doc_id in self.tokenized_documents:
            score = self.score(query_tokens, doc_id)

            if score > 0:
                results.append((doc_id, score))

        results.sort(key=lambda x: x[1], reverse=True)

        return query_tokens, results[:top_k]
```
Si noti che questa implementazione è molto inefficiente perché il metodo rank calcola lo score per tutti i documenti indipendentemente dal fatto che contengano o meno i termini della query. In una implementazione migliore si userebbe un indice invertito per calcolare lo score solo sui documenti che contengono almeno uno dei termini della query (OR).

**NB** abbiamo tralasciato in questi ragionamenti la term frequency della query. Questo è ragionevole per query brevi, come quelle tipiche dei motori di ricerca. Però per query lunghe e descrizioni più articolate si può introdurre un fattore di **query term frequency**, controllato con un parametro $k_3$ (vedilo nella teoria).

$k_1$ va da 0 a $\infty$ e controlla la saturazione della TF, mentre $b$ va da 0 a 1 e controlla la normalizzazione della lunghezza del documento ($b=0$ non normalizza per la lunghezza, $b=1$ normalizzazione piena).

Tipici valori dei parametri: 
$$k_1 \in [1.2, 2.0] \qquad b \approx 0.75$$
però questi valori non sono leggi universali e in un sistema reali devono essere testati tramite grid search su un development set, basandosi su metriche di valutazione come precision, recall, MAP, NDCG, MRR etc...

## Language Models (LM)
L'altra famiglia importante di modelli di scoring probabilistici oltre a BM25 è quella dei Language Models (LM).

Mentre BM25 si chiede informalmente quanta evidenza forniscono i termini della query a favore della rilevanza del documento, i language models si chidedono invece **quanto è probabile che la query sia stata generata dal modello linguistico del documento**.

Formalmente, per un documento $d$ costruiamo un modello $M_d$ e ordiniamo i documenti secondo $p(q | M_d)$, cioè la probabilità che la query $q$ sia stata generata dal modello linguistico del documento $d$.

Con una query composta dai termini $w_1, w_2, ..., w_m$ utilizzeremo il language model unigramma:
$$p(q | M_d) = \prod_{i=1}^m p(w_i | M_d)$$
per evitare il problema delle probabilità molto piccole e quindi underflow, useremo i logaritmi:
$$\log p(q | M_d) = \sum_{i=1}^m \log p(w_i | M_d)$$

Come visto nella teoria, è necessario per LM introdurre tecniche di smoothing per evitare che termini della query che non compaiono nel documento facciano crollare lo score a zero.

Nel laboratorio implementiamo lo smoothing di Dirichlet:
$$p(t | d) = \frac{tf_{t,d} + \mu p(t | C)}{|d| + \mu}$$
dove $p(t | C)$ è la probabilità del termine $t$ nella collezione mentre $\mu$ è un parametro che controlla l'effetto dello smoothing. 

Possiamo leggere intuitivamente la formula così: 
- il documento contribuisce allo score con il conteggio osservato $tf_{t,d}$
- la collezione fornisce un contributo di smoothing "background" $p(t | C)$, il cui peso è controllato dal parametro $\mu$.

Se $\mu$ è piccolo, il modello si basa principalmente sui dati osservati nel documento, mentre se $\mu$ è grande, il modello del documento assomiglia di più al modello della collezione, quindi lo score dipende principalmente dalla probabilità del termine nella collezione.


Nel LM, a differenza di BM25 base che abbiamo implementato sopra, **si mantengono le eventuali ripetizioni dei termini nella query** (non si considera solo l'insieme dei termini distinti della query):
$$\log p(q | M_d) = \sum_{t \in q} \log p(t | d)$$
Quindi se un termine compare due volte nella query, contribuisce due volte alla log-likelihood.

Nota: Se un termine della query non compare mai nella collezione, il modello della collezione assegna probabilità zero. In questo notebook ignoriamo questi termini fuori vocabolario, perché non aiutano a distinguere i documenti della collezione. In un sistema reale, la gestione degli out-of-vocabulary terms richiede scelte più attente.

```python
class QueryLikelihoodDirichletLM:
    """
    Language Model per IR con query likelihood e Dirichlet smoothing.

    Parameters
    ----------
    mu : float
        Parametro di smoothing Dirichlet.
    """

    def __init__(self, mu=1000.0):
        self.mu = mu

    def fit(self, tokenized_documents):
        self.tokenized_documents = tokenized_documents
        self.N = len(tokenized_documents)

        self.doc_len = {}
        self.term_frequencies = {}
        self.collection_frequency = Counter()
        self.collection_length = 0

        for doc_id, tokens in tokenized_documents.items():
            self.doc_len[doc_id] = len(tokens)

            tf = Counter(tokens)
            self.term_frequencies[doc_id] = tf

            self.collection_frequency.update(tokens)
            self.collection_length += len(tokens)

        return self

    def collection_probability(self, term):
        """
        Probabilità del termine nel modello della collezione.
        """
        cf = self.collection_frequency.get(term, 0)

        if self.collection_length == 0:
            return 0.0

        return cf / self.collection_length

    def smoothed_term_probability(self, term, doc_id):
        """
        Probabilità smoothed del termine nel documento.
        """
        tf = self.term_frequencies[doc_id].get(term, 0)
        doc_len = self.doc_len[doc_id]
        p_collection = self.collection_probability(term)

        return (tf + self.mu * p_collection) / (doc_len + self.mu)

    def score(self, query_tokens, doc_id):
        """
        Log-likelihood della query dato il documento.

        I termini fuori vocabolario vengono ignorati.
        """
        score = 0.0
        used_terms = 0

        for term in query_tokens:
            p_collection = self.collection_probability(term)

            if p_collection == 0.0:
                # Termine fuori vocabolario: non aiuta a distinguere i documenti,
                # quindi in questo notebook lo ignoriamo.
                continue

            p = self.smoothed_term_probability(term, doc_id)
            score += math.log(p)
            used_terms += 1

        if used_terms == 0:
            return float("-inf")

        return score

    def rank(self, query, top_k=10):
        query_tokens = preprocess(query, is_query=True)

        results = []

        for doc_id in self.tokenized_documents:
            score = self.score(query_tokens, doc_id)

            if score != float("-inf"):
                results.append((doc_id, score))

        results.sort(key=lambda x: x[1], reverse=True)

        return query_tokens, results[:top_k]
```

Si osserva come gli score di LM siano negativi: questo è dovuto al fatto che stiamo usando i logaritmi delle probabilità, che sono sempre negativi o al massimo zero (quando la probabilità è 1). Quindi in LM, score più alto (meno negativo) indica documenti più rilevanti.

Come anticipato, $\mu$ controlla la forza dello smoothing. Più $\mu$ è piccolo, più il modello si fida del documento. Più $\mu$ è grande, più il modello si fida della collezione. 

Osservando il ranking al variare dei valori di $\mu$, si nota come un $\mu$ più piccolo porti a documenti più corti e più specifici (con TF più evidenti), mentre un $\mu$ più grande porti a documenti più lunghi e più generali (con TF meno evidenti). Questo perché quando $\mu$ è piccolo la probabilità dei termini nella collezione è meno influente --> sono i documenti specifici che effettivamente contengono spesso i termini della query a essere favoriti, mentre quando $\mu$ è grande la probabilità dei termini nella collezione è più influente --> sono i documenti più generali che contengono molti termini della query, anche se con TF meno evidenti, a essere favoriti.

Formula in forma classica:
$$\log p(q | M_d) = \sum_{t \in q} \log p(t | M_d)$$
es. query "bm25 bm25 retrieval" --> $log p("bm25" | M_d) + log p("bm25" | M_d) + log p("retrieval" | M_d)$

Formula in forma bag-of-words (usando i termini distinti della query):
$$\log p(q | M_d) = \sum_{t \in q^*} tf_{t,d} \cdot \log p(t | M_d)$$

A prima vista, a differenza di BM25, in LM **sembra mancare un termine esplicito tipo IDF**. Può sembrare quindi che un termine molto comune, come the, sia favorito dai LM perché ha probabilità alta. Però in realtà usando uno smoothing come Dirichlet:
$$p(t | d) = \frac{tf_{t,d} + \mu p(t | C)}{|d| + \mu}$$
si ha che il background della collezione $p(t | C)$ è grande, ma tutti i documenti lo ricevono allo stesso modo. 

Il punto cruciale inoltre è che in LM il ranking in realtà **dipende soprattutto da quanto il documento usa il termine più della collezione**. Riscrivendo la formula infatti:

<img src="img/lm_idf.png" alt="LM IDF" width="400">

il secondo termine presenta il rapporto 
$$\frac{tf_{t,d}}{\mu p(t | C)}$$
Se un termine appare molto nella collezione ma relativamente poco nel documento, allora questo rapporto è piccolo e quindi il contributo totale è piccolo. Al contrario se il termine appare molto più nel documento che nella collezione, allora questo rapporto è grande e quindi il contributo totale è grande. 

Attenzione però al fatto che lo scoring completo contiene anche al denominatore 
$$|d| + \mu$$
Quindi:
$$\log p(t | d) = \log \left( {tf_{t,d} + \mu p(t | C)} \right) - \log (|d| + \mu)$$
Per una query con $|q|$ termini:
$$\log p(q | d) = - |q| \cdot \log (|d| + \mu) + \sum_{t \in q} \log \left( {tf_{t,d} + \mu p(t | C)} \right) $$
quindi il denominatore $|d| + \mu$ introduce anche una **normalizzazione legata alla lunghezza del documento**

In sintesi:
- i termini comuni hanno probabilità alta nel background della collezione, quindi aggiungono poca evidenza specifica a favore dei documenti che li contengono come visto nella prima parte di questa sezione
- i termini rari al contrario hanno probabilità bassa nel background --> rapporto $\frac{tf_{t,d}}{\mu p(t | C)}$ più alto --> più evidenza specifica a favore dei documenti che li contengono
- Dirichlet smoothing introduce anche una normalizzazione rispetto alla lunghezza del documento